In [ ]:
# %%
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from pathlib import Path
from IPython.display import display
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    accuracy_score,
    balanced_accuracy_score
)

# =========================
# Configuración general
# =========================

DATA_DIR = Path("data")
FIG_DIR = DATA_DIR / "graficos"
FIG_DIR.mkdir(parents=True, exist_ok=True)

def buscar_csv(nombre):
    """Busca primero en data/ y después en el directorio actual."""
    for ruta in (DATA_DIR / nombre, Path(nombre)):
        if ruta.exists():
            return ruta
    return DATA_DIR / nombre

CSV_PATH = buscar_csv("resultados_comparativa_llm.csv")

UMBRAL_FASE1 = 0.25  # umbral elegido en dev set, escala 0-1

sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "serif",
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "legend.title_fontsize": 10,
    "axes.titlesize": 12
})

PALETA_CLASES = {
    "no_plagio": "#B22222",
    "plagio": "#2F5597"
}

ORDEN_CLASES = ["no_plagio", "plagio"]


# %%
# =========================
# Carga y preparación
# =========================

df = pd.read_csv(CSV_PATH)

def normalizar_etiqueta(x):
    x = str(x).strip().lower()
    x = x.replace(" ", "_")

    if "no_plagio" in x or "no-plagio" in x:
        return "no_plagio"
    elif "plagio" in x:
        return "plagio"
    else:
        return "desconocido"


def convertir_bool(x):
    if isinstance(x, bool):
        return x

    x = str(x).strip().lower()

    if x in ["true", "1", "sí", "si", "yes"]:
        return True
    elif x in ["false", "0", "no"]:
        return False
    else:
        return np.nan


def ordenar_nivel(nivel):
    try:
        return int(str(nivel).split("_")[-1])
    except:
        return 999


def nombre_seguro(nombre):
    return (
        str(nombre)
        .replace(":", "_")
        .replace(".", "_")
        .replace("/", "_")
        .replace(" ", "_")
    )


def guardar_figura(nombre):
    plt.savefig(FIG_DIR / f"{nombre}.png", dpi=300, bbox_inches="tight")


def calcular_metricas(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    especificidad = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    return {
        "N": len(y_true),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced accuracy": balanced_accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Especificidad": especificidad,
        "F0.5": fbeta_score(y_true, y_pred, beta=0.5, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "F2": fbeta_score(y_true, y_pred, beta=2, zero_division=0)
    }


df["etiqueta_real_norm"] = df["etiqueta_real"].apply(normalizar_etiqueta)
df["y_true"] = (df["etiqueta_real_norm"] == "plagio").astype(int)

modelos = [
    c.replace("prediccion_", "")
    for c in df.columns
    if c.startswith("prediccion_")
]

# Similitud en porcentaje
df["sim_maxima_num"] = pd.to_numeric(df["sim_maxima"], errors="coerce")
df["sim_levenshtein_num"] = pd.to_numeric(df["sim_levenshtein"], errors="coerce")
df["sim_trigramas_num"] = pd.to_numeric(df["sim_trigramas"], errors="coerce")

if df["sim_maxima_num"].max() <= 1.5:
    df["sim_maxima_pct"] = df["sim_maxima_num"] * 100
    umbral_pct = UMBRAL_FASE1 * 100
else:
    df["sim_maxima_pct"] = df["sim_maxima_num"]
    umbral_pct = UMBRAL_FASE1 * 100

if df["sim_levenshtein_num"].max() <= 1.5:
    df["sim_levenshtein_pct"] = df["sim_levenshtein_num"] * 100
else:
    df["sim_levenshtein_pct"] = df["sim_levenshtein_num"]

if df["sim_trigramas_num"].max() <= 1.5:
    df["sim_trigramas_pct"] = df["sim_trigramas_num"] * 100
else:
    df["sim_trigramas_pct"] = df["sim_trigramas_num"]

# Predicción de la fase 1
df["pred_fase1"] = df["sim_maxima_pct"] >= umbral_pct

# Si existe la columna ya calculada, se comprueba coherencia
if "pasa_filtro_matematico" in df.columns:
    df["pasa_filtro_matematico_bool"] = df["pasa_filtro_matematico"].apply(convertir_bool)

    diferencias = (
        df["pred_fase1"] != df["pasa_filtro_matematico_bool"]
    ).sum()

    if diferencias == 0:
        df["pred_fase1"] = df["pasa_filtro_matematico_bool"]
    else:
        print(f"Aviso: hay {diferencias} diferencias entre el umbral calculado y pasa_filtro_matematico.")


df["nivel"] = pd.Categorical(
    df["nivel"],
    categories=sorted(df["nivel"].dropna().unique(), key=ordenar_nivel),
    ordered=True
)

display(df.head())
print("Modelos detectados:", modelos)

In [ ]:
# %%
# =========================================================
# FASE 1: filtro léxico basado en similitud
# =========================================================

y_true = df["y_true"].astype(int)
y_pred_fase1 = df["pred_fase1"].astype(int)

metricas_fase1 = calcular_metricas(y_true, y_pred_fase1)

tabla_fase1_metricas = pd.DataFrame([{
    "Fase": "Filtro léxico",
    **metricas_fase1
}])

columnas_fase1 = [
    "Fase", "N", "TN", "FP", "FN", "TP",
    "Precision", "Recall", "Especificidad",
    "F2", "F1", "Accuracy", "Balanced accuracy"
]

tabla_fase1_metricas = tabla_fase1_metricas[columnas_fase1]

display(
    tabla_fase1_metricas.style.format({
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "Especificidad": "{:.3f}",
        "F2": "{:.3f}",
        "F1": "{:.3f}",
        "Accuracy": "{:.3f}",
        "Balanced accuracy": "{:.3f}"
    })
)



# %%
# Matriz de confusión de la fase 1

cm_fase1 = confusion_matrix(y_true, y_pred_fase1, labels=[0, 1])

tabla_cm_fase1 = pd.DataFrame(
    cm_fase1,
    index=["Real: no plagio", "Real: plagio"],
    columns=["Predicción: no plagio", "Predicción: plagio"]
)

display(tabla_cm_fase1)

tabla_cm_fase1_reset = tabla_cm_fase1.reset_index().rename(columns={"index": "Clase real"})


# %%
# Tabla por nivel para la fase 1

filas_nivel_fase1 = []

for nivel, g in df.groupby("nivel", observed=True):
    yt = g["y_true"].astype(int)
    yp = g["pred_fase1"].astype(int)

    tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    tasa_fp = fp / (fp + tn) if (fp + tn) > 0 else np.nan
    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan

    filas_nivel_fase1.append({
        "Nivel": nivel,
        "N": len(g),
        "Plagios reales": int(yt.sum()),
        "No plagios reales": int((yt == 0).sum()),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision,
        "Recall": recall,
        "Tasa FP": tasa_fp
    })

tabla_fase1_nivel = pd.DataFrame(filas_nivel_fase1)

display(
    tabla_fase1_nivel.style.format({
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "Tasa FP": "{:.3f}"
    })
)



In [ ]:
# %%
# Distribución de similitud máxima por clase real
PALETA_METRICAS = {
    "Precision": "#08306B",
    "Recall": "#2171B5",
    "F0.5": "#4292C6",
    "F1": "#6BAED6",
    "F2": "#9ECAE1"
}

plt.figure(figsize=(8, 5))

sns.histplot(
    data=df,
    x="sim_maxima_pct",
    hue="etiqueta_real_norm",
    hue_order=ORDEN_CLASES,
    bins=np.linspace(0, 100, 21),
    stat="density",
    common_norm=False,
    element="step",
    fill=True,
    alpha=0.30,
    palette=PALETA_CLASES
)

plt.axvline(
    umbral_pct,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label=f"Umbral ({umbral_pct:.0f}%)"
)

plt.xlabel("Similitud máxima (%)")
plt.ylabel("Densidad")
plt.xlim(0, 100)
plt.legend(title="Etiqueta real")

plt.tight_layout()
guardar_figura("fase1_distribucion_similitud")
plt.show()


# %%
# Matriz de confusión de fase 1

plt.figure(figsize=(4.8, 4))

sns.heatmap(
    cm_fase1,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["No plagio", "Plagio"],
    yticklabels=["No plagio", "Plagio"]
)

plt.xlabel("Predicción")
plt.ylabel("Etiqueta real")

plt.tight_layout()
guardar_figura("fase1_matriz_confusion")
plt.show()


In [ ]:
# %%
# Distribución de las métricas léxicas: Levenshtein y trigramas en paralelo

metricas_lexicas = [
    ("sim_levenshtein_pct", "Similitud de Levenshtein (%)"),
    ("sim_trigramas_pct", "Similitud de trigramas (%)")
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for ax, (columna, etiqueta_x) in zip(axes, metricas_lexicas):
    sns.histplot(
        data=df,
        x=columna,
        hue="etiqueta_real_norm",
        hue_order=ORDEN_CLASES,
        bins=np.linspace(0, 100, 21),
        stat="density",
        common_norm=False,
        element="step",
        fill=True,
        alpha=0.30,
        palette=PALETA_CLASES,
        ax=ax
    )

    ax.axvline(
        umbral_pct,
        color="black",
        linestyle="--",
        linewidth=1.5,
        label=f"Umbral ({umbral_pct:.0f}%)"
    )

    ax.set_xlabel(etiqueta_x)
    ax.set_xlim(0, 100)
    ax.legend(title="Etiqueta real")

axes[0].set_ylabel("Densidad")
axes[1].set_ylabel("")

plt.tight_layout()
guardar_figura("fase1_distribucion_metricas_lexicas")
plt.show()


In [ ]:
# %%
# =========================================================
# FASE 2: modelos LLM sobre los candidatos filtrados
# =========================================================
PALETA_METRICAS = {
    "Precision": "#4C78A8",   # azul apagado
    "Recall": "#59A14F",      # verde apagado
    "F0.5": "#B07AA1",        # morado suave
    "F1": "#F28E2B",          # naranja suave
    "F2": "#9C755F"           # marrón suave
}

df_fase2 = df[df["pred_fase1"] == True].copy()
y_true_fase2 = df_fase2["y_true"].astype(int)

filas_fase2 = []

for m in modelos:
    col_pred = f"prediccion_{m}"
    col_tiempo = f"tiempo_{m}"

    y_pred_modelo = (
        df_fase2[col_pred]
        .apply(normalizar_etiqueta)
        .eq("plagio")
        .astype(int)
    )

    met = calcular_metricas(y_true_fase2, y_pred_modelo)

    tiempo_medio = pd.to_numeric(df_fase2[col_tiempo], errors="coerce").mean()
    tiempo_mediano = pd.to_numeric(df_fase2[col_tiempo], errors="coerce").median()

    filas_fase2.append({
        "Modelo": m,
        **met,
        "Tiempo medio (s)": tiempo_medio,
        "Tiempo mediano (s)": tiempo_mediano
    })

tabla_fase2_metricas = pd.DataFrame(filas_fase2)

tabla_fase2_metricas = tabla_fase2_metricas.sort_values(
    by=["Precision", "F0.5", "F1", "Recall"],
    ascending=False
).reset_index(drop=True)

columnas_fase2 = [
    "Modelo", "N", "TN", "FP", "FN", "TP",
    "Precision", "Recall", "Especificidad",
    "F0.5", "F1", "Accuracy", "Balanced accuracy",
    "Tiempo medio (s)", "Tiempo mediano (s)"
]

tabla_fase2_metricas = tabla_fase2_metricas[columnas_fase2]

display(
    tabla_fase2_metricas.style.format({
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "Especificidad": "{:.3f}",
        "F0.5": "{:.3f}",
        "F1": "{:.3f}",
        "Accuracy": "{:.3f}",
        "Balanced accuracy": "{:.3f}",
        "Tiempo medio (s)": "{:.2f}",
        "Tiempo mediano (s)": "{:.2f}"
    })
)


top_3_fase2 = tabla_fase2_metricas["Modelo"].head(3).tolist()
best_model_fase2 = top_3_fase2[0]

print("Mejor modelo fase 2:", best_model_fase2)
print("Top 3 fase 2:", top_3_fase2)

In [ ]:
# %%
# Métricas de fase 2 por modelo

metricas_plot_fase2 = tabla_fase2_metricas.melt(
    id_vars="Modelo",
    value_vars=["Precision", "Recall", "F0.5", "F1"],
    var_name="Métrica",
    value_name="Valor"
)

plt.figure(figsize=(10, 5))

sns.barplot(
    data=metricas_plot_fase2,
    x="Modelo",
    y="Valor",
    hue="Métrica",
    order=tabla_fase2_metricas["Modelo"],
    palette=PALETA_METRICAS
)

plt.xlabel("Modelo")
plt.ylabel("Valor de la métrica")
plt.ylim(0, 1.05)
plt.xticks(rotation=35, ha="right")
plt.legend(title="Métrica", loc="lower right")

plt.tight_layout()
guardar_figura("fase2_metricas_modelos")
plt.show()


# %%
# Relación entre precisión y tiempo medio

df_tradeoff_fase2 = tabla_fase2_metricas[[
    "Modelo", "Precision", "Recall", "F0.5", "F1", "Tiempo medio (s)"
]].copy()

plt.figure(figsize=(8, 5))

sns.scatterplot(
    data=df_tradeoff_fase2,
    x="Tiempo medio (s)",
    y="Precision",
    s=120,
    color="#4C78A8"
)

for _, row in df_tradeoff_fase2.iterrows():
    plt.text(
        row["Tiempo medio (s)"] * 1.02,
        row["Precision"],
        row["Modelo"],
        fontsize=9,
        va="center"
    )

plt.xlabel("Tiempo medio de ejecución (s)")
plt.ylabel("Precisión")
plt.ylim(max(0, df_tradeoff_fase2["Precision"].min() - 0.1), 1.05)
plt.xlim(0, df_tradeoff_fase2["Tiempo medio (s)"].max() * 1.30)

plt.tight_layout()
guardar_figura("fase2_precision_tiempo")
plt.show()


In [ ]:
# %%
# =========================================================
# SISTEMA COMPLETO: fase 1 + fase 2
# =========================================================

filas_sistema = []

for m in modelos:
    pred_llm = (
        df[f"prediccion_{m}"]
        .apply(normalizar_etiqueta)
        .eq("plagio")
        .astype(int)
    )

    # Si no pasa fase 1, el sistema final predice no plagio.
    # Si pasa fase 1, se usa la predicción del LLM.
    pred_final = np.where(df["pred_fase1"], pred_llm, 0).astype(int)

    df[f"pred_final_{m}"] = pred_final

    met = calcular_metricas(y_true, pred_final)

    filas_sistema.append({
        "Modelo": m,
        **met
    })

tabla_sistema_metricas = pd.DataFrame(filas_sistema)

tabla_sistema_metricas = tabla_sistema_metricas.sort_values(
    by=["Precision", "F0.5", "F1", "Recall"],
    ascending=False
).reset_index(drop=True)

columnas_sistema = [
    "Modelo", "N", "TN", "FP", "FN", "TP",
    "Precision", "Recall", "Especificidad",
    "F0.5", "F1", "F2", "Accuracy", "Balanced accuracy"
]

tabla_sistema_metricas = tabla_sistema_metricas[columnas_sistema]

display(
    tabla_sistema_metricas.style.format({
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "Especificidad": "{:.3f}",
        "F0.5": "{:.3f}",
        "F1": "{:.3f}",
        "F2": "{:.3f}",
        "Accuracy": "{:.3f}",
        "Balanced accuracy": "{:.3f}"
    })
)


top_3_sistema = tabla_sistema_metricas["Modelo"].head(3).tolist()
best_model_sistema = top_3_sistema[0]

print("Mejor modelo sistema completo:", best_model_sistema)
print("Top 3 sistema completo:", top_3_sistema)

In [ ]:
# %%
# Matrices de confusión del sistema completo para los tres mejores modelos

fig, axes = plt.subplots(1, len(top_3_sistema), figsize=(5 * len(top_3_sistema), 4))

if len(top_3_sistema) == 1:
    axes = [axes]

for ax, m in zip(axes, top_3_sistema):
    y_pred_final = df[f"pred_final_{m}"].astype(int)

    cm = confusion_matrix(y_true, y_pred_final, labels=[0, 1])

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        cbar=False,
        ax=ax,
        xticklabels=["No plagio", "Plagio"],
        yticklabels=["No plagio", "Plagio"]
    )

    ax.set_title(m)
    ax.set_xlabel("Predicción")
    ax.set_ylabel("Etiqueta real")

plt.tight_layout()
guardar_figura("sistema_completo_matrices_confusion_top3")
plt.show()


# %%
# Comparación entre fase 1 y sistema completo con el mejor modelo

met_fase1 = calcular_metricas(y_true, y_pred_fase1)

met_sistema_best = calcular_metricas(
    y_true,
    df[f"pred_final_{best_model_sistema}"].astype(int)
)

tabla_comparacion_fases = pd.DataFrame([
    {
        "Evaluación": "Fase 1",
        "Modelo": "Filtro léxico",
        **met_fase1
    },
    {
        "Evaluación": "Sistema completo",
        "Modelo": best_model_sistema,
        **met_sistema_best
    }
])

tabla_comparacion_fases = tabla_comparacion_fases[[
    "Evaluación", "Modelo", "N", "TN", "FP", "FN", "TP",
    "Precision", "Recall", "Especificidad",
    "F0.5", "F1", "F2", "Accuracy", "Balanced accuracy"
]]

display(
    tabla_comparacion_fases.style.format({
        "Precision": "{:.3f}",
        "Recall": "{:.3f}",
        "Especificidad": "{:.3f}",
        "F0.5": "{:.3f}",
        "F1": "{:.3f}",
        "F2": "{:.3f}",
        "Accuracy": "{:.3f}",
        "Balanced accuracy": "{:.3f}"
    })
)


# %%
# Comparación visual entre fase 1 y sistema completo

plot_comparacion = tabla_comparacion_fases.melt(
    id_vars=["Evaluación", "Modelo"],
    value_vars=["Precision", "Recall", "F0.5", "F1"],
    var_name="Métrica",
    value_name="Valor"
)

plt.figure(figsize=(8, 5))

sns.barplot(
    data=plot_comparacion,
    x="Métrica",
    y="Valor",
    hue="Evaluación",
    palette=["#BFBFBF", "#4C78A8"]
)

plt.xlabel("Métrica")
plt.ylabel("Valor")
plt.ylim(0, 1.05)
plt.legend(title="Evaluación", loc="lower right")

plt.tight_layout()
guardar_figura("comparacion_fase1_sistema_completo")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

# ==========================================================
# Configuración general
# ==========================================================

CSV_PATH = buscar_csv("resultados_clough_ancho.csv")

UMBRAL = 0.25
UMBRAL_PCT = UMBRAL * 100

MODELO_FINAL = "poligpt"   # Cambia aquí el modelo si quieres usar otro

df = pd.read_csv(CSV_PATH)

# ==========================================================
# Estética común para TFG
# ==========================================================

plt.rcParams.update({
    "font.family": "serif",
    "mathtext.fontset": "cm",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10.5,
    "ytick.labelsize": 10.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": "--",
    "figure.dpi": 120,
})

COLOR_PLAGIO = "#4C78A8"
COLOR_NO_PLAGIO = "#D95F02"
COLOR_GRIS = "#6E6E6E"
COLOR_UMBRAL = "#333333"

orden_categorias = ["cut", "light", "heavy", "non"]
orden_categorias = [c for c in orden_categorias if c in df["nivel"].unique()]

nombres_categorias = {
    "cut": "Cut",
    "light": "Light",
    "heavy": "Heavy",
    "non": "Non"
}

# ==========================================================
# Funciones auxiliares
# ==========================================================

def normalizar_etiqueta(x):
    x = str(x).strip().lower()
    if x in ["plagio", "1", "true", "si", "sí"]:
        return "plagio"
    if x in ["no_plagio", "no plagio", "0", "false", "no"]:
        return "no_plagio"
    return x


def metricas_binarias(y_true, y_pred):
    y_true = pd.Series(y_true).apply(normalizar_etiqueta)
    y_pred = pd.Series(y_pred).apply(normalizar_etiqueta)

    tp = int(((y_true == "plagio") & (y_pred == "plagio")).sum())
    tn = int(((y_true == "no_plagio") & (y_pred == "no_plagio")).sum())
    fp = int(((y_true == "no_plagio") & (y_pred == "plagio")).sum())
    fn = int(((y_true == "plagio") & (y_pred == "no_plagio")).sum())

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    return {
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Accuracy": accuracy,
    }


def prediccion_sistema_completo(df, modelo):
    pred_col = f"prediccion_{modelo}"

    if pred_col not in df.columns:
        raise ValueError(f"No existe la columna {pred_col}. Revisa el nombre del modelo.")

    pred_modelo = df[pred_col].apply(normalizar_etiqueta)

    pred_final = np.where(
        df["pasa_filtro_matematico"],
        pred_modelo,
        "no_plagio"
    )

    return pd.Series(pred_final, index=df.index)


def finalizar_figura(nombre=None):
    plt.tight_layout()

    if nombre is not None:
        plt.savefig(FIG_DIR / f"{nombre}.png", dpi=300, bbox_inches="tight")

    plt.show()


df["etiqueta_real"] = df["etiqueta_real"].apply(normalizar_etiqueta)

# En tu CSV sim_maxima ya está en porcentaje
df["sim_pct"] = df["sim_maxima"]

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 4.4))

datos_boxplot = [
    df.loc[df["nivel"] == cat, "sim_pct"].dropna()
    for cat in orden_categorias
]

bp = ax.boxplot(
    datos_boxplot,
    labels=[nombres_categorias[c] for c in orden_categorias],
    patch_artist=True,
    widths=0.55,
    showfliers=True,
    medianprops={"color": "black", "linewidth": 1.4},
    boxprops={"linewidth": 1.1},
    whiskerprops={"linewidth": 1.1},
    capprops={"linewidth": 1.1},
    flierprops={"marker": "o", "markersize": 4, "alpha": 0.55},
)

for patch in bp["boxes"]:
    patch.set_facecolor("#D9E2EC")
    patch.set_edgecolor("#3A3A3A")

ax.axhline(
    UMBRAL_PCT,
    linestyle=":",
    linewidth=1.6,
    color=COLOR_UMBRAL,
    label=f"Umbral = {UMBRAL_PCT:.0f}%"
)

ax.set_xlabel("Categoría del corpus")
ax.set_ylabel("Similitud máxima")
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
ax.set_ylim(0, 105)
ax.legend(frameon=True, loc="upper right")

finalizar_figura("clough_distribucion_similitud_categoria")

In [ ]:
tasas = (
    df.groupby("nivel")["pasa_filtro_matematico"]
    .mean()
    .reindex(orden_categorias)
    * 100
)

fig, ax = plt.subplots(figsize=(7.2, 4.4))

leyendas_usadas = set()

for i, cat in enumerate(orden_categorias):
    etiqueta = df.loc[df["nivel"] == cat, "etiqueta_real"].mode().iloc[0]

    if etiqueta == "plagio":
        color = COLOR_PLAGIO
        label = "Plagio detectado"
    else:
        color = COLOR_NO_PLAGIO
        label = "No plagio marcado"

    if label in leyendas_usadas:
        label = None
    else:
        leyendas_usadas.add(label)

    ax.bar(
        i,
        tasas.loc[cat],
        color=color,
        edgecolor="black",
        linewidth=0.8,
        width=0.62,
        label=label
    )

    ax.text(
        i,
        tasas.loc[cat] + 2.2,
        f"{tasas.loc[cat]:.1f}%",
        ha="center",
        va="bottom",
        fontsize=10
    )

ax.set_xticks(range(len(orden_categorias)))
ax.set_xticklabels([nombres_categorias[c] for c in orden_categorias])
ax.set_xlabel("Categoría del corpus")
ax.set_ylabel("Respuestas marcadas como sospechosas")
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
ax.set_ylim(0, 110)
ax.legend(frameon=True, loc="upper right")

finalizar_figura("clough_tasa_sospechosas_categoria")

In [ ]:
pred_final = prediccion_sistema_completo(df, MODELO_FINAL)
metricas_clough = metricas_binarias(df["etiqueta_real"], pred_final)

cm = np.array([
    [metricas_clough["TN"], metricas_clough["FP"]],
    [metricas_clough["FN"], metricas_clough["TP"]],
])

fig, ax = plt.subplots(figsize=(4.8, 4.3))

im = ax.imshow(cm, cmap="Blues")

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["No plagio", "Plagio"])
ax.set_yticklabels(["No plagio", "Plagio"])

ax.set_xlabel("Predicción del sistema")
ax.set_ylabel("Etiqueta real")

for i in range(2):
    for j in range(2):
        color_texto = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center",
            fontsize=15,
            color=color_texto
        )

ax.grid(False)
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.ax.set_ylabel("Número de casos", rotation=270, labelpad=14)

finalizar_figura(f"clough_matriz_confusion_{MODELO_FINAL.replace(':', '_')}")

In [ ]:
metricas_sintetico = {
    "Precision": 1.000,
    "Recall": 0.979,
    "F1": 0.989,
}

metricas_comparacion = ["Precision", "Recall", "F1"]

valores_sintetico = [metricas_sintetico[m] * 100 for m in metricas_comparacion]
valores_clough = [metricas_clough[m] * 100 for m in metricas_comparacion]

x = np.arange(len(metricas_comparacion))
width = 0.34

fig, ax = plt.subplots(figsize=(7.2, 4.4))

bars1 = ax.bar(
    x - width / 2,
    valores_sintetico,
    width,
    label="Conjunto sintético",
    color=COLOR_PLAGIO,
    edgecolor="black",
    linewidth=0.8
)

bars2 = ax.bar(
    x + width / 2,
    valores_clough,
    width,
    label="Corpus de Clough",
    color=COLOR_GRIS,
    edgecolor="black",
    linewidth=0.8
)

ax.set_xticks(x)
ax.set_xticklabels(["Precision", "Recall", r"$F_1$"])
ax.set_ylabel("Valor de la métrica")
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100))
ax.set_ylim(0, 110)
ax.legend(frameon=True, loc="lower right")

for bars in [bars1, bars2]:
    for bar in bars:
        altura = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            altura + 2,
            f"{altura:.1f}%",
            ha="center",
            va="bottom",
            fontsize=9.5
        )

finalizar_figura(f"comparacion_sintetico_clough_{MODELO_FINAL.replace(':', '_')}")